# 🏠 حل مشروع التنبؤ بأسعار العقارات (House Prices Solution)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/house_prices_data.csv')
print(df.head())

### 🛠️ الجزء الأول: تحويل التوزيعات وعلاج القيم المفقودة

In [ ]:
# Fill missing square_footage with neighborhood median
df['square_footage'] = df.groupby('neighborhood')['square_footage'].transform(lambda x: x.fillna(x.median()))

# Log transform target
df['log_price'] = np.log1p(df['sale_price'])
print("Missing count:", df.isna().sum().sum())

### 🤖  الجزء الثاني: مقارنة خوارزميات الانحدار وتوليف المعاملات

In [ ]:
X = df.drop(columns=['house_id', 'sale_price', 'log_price'])
y = df['log_price']

X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    # Transform back to original scale to compute actual RMSE
    actual_preds = np.expm1(preds)
    actual_y_test = np.expm1(y_test)
    rmse = np.sqrt(mean_squared_error(actual_y_test, actual_preds))
    r2 = r2_score(actual_y_test, actual_preds)
    results[name] = {"RMSE": rmse, "R2": r2}
    print(f"{name} - Test RMSE: ${rmse:.2f}, R2 Score: {r2:.4f}")

# Plot Comparison
res_df = pd.DataFrame(results).T
res_df['R2'].plot(kind='bar', color='skyblue', figsize=(8, 5))
plt.title('Model R2 Comparison (House Prices)')
plt.ylabel('R2 Score')
plt.tight_layout()
plt.savefig('solutions/house_model_comparison.png')
plt.show()